# Graduation Rate Risk Analysis

This notebook explores graduation rate outcomes across Florida schools, districts, and structrual categories to identify early signals associated with elevated graduation risk.

The analysis focuses on distributional patterns and subgroup comparisons rather than formal model construction.

## 1. Lowest Observed Graduation Outcomes

Graduation rates are first examined at the school level to surface extreme outcomes and validate the integrity of the outcome variable.

Identifying the lowest-performing schools provides context for subsequent aggregation and stratified analysis.

In [ ]:
import duckdb

con = duckdb.connect("../accountability_project/dev.duckdb")

query = """
select
    district_number,
    district_name,
    school_number,
    school_name,
    graduation_rate_2023_24
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
order by graduation_rate_2023_24 asc
limit 10
"""

con.execute(query).fetchdf()

## 2. District-Level Aggregation

To determine whether low graduation outcomes are isolated or concentrated, graduation rates are aggregated at the district level.

Average graduation rates by district highlight broader performance patterns that may not be visible at the individual school level.

In [ ]:
query = """
select
    district_number,
    district_name,
    avg(graduation_rate_2023_24) as avg_grade_rate
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by district_number, district_name
order by avg_grade_rate asc
limit 5
"""

con.execute(query).fetchdf()

## 3. Graduation Rates by School Type

Graduation rates are next examined across school types to assess whether structrual differences are associated with outcome variation.

This aggregation provides a high-level comparison of average performance and sample size across school categories.

In [ ]:
query = """
select
    school_type,
    avg(graduation_rate_2023_24) as avg_grade_rate,
    count(*) as n_schools
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type
order by avg_grade_rate asc
limit 5
"""

con.execute(query).fetchdf()

## 4. Outcome Coverage and Missingness

Not all school types rport graduation rates. To understand the analytic population, the distribution of missing graduation outcomes is examined by school type.

This step clarifies which subsets of schools are represented in graduation rate analyses and which are excluded by design.

In [ ]:
query = """
select
    school_type,
    count(*) as n_rows,
    sum(case when graduation_rate_2023_24 is null then 1 else 0 end) as n_null_grad_rate
from main.stg_fl_school_grades
group by 1
order by 1;
"""

con.execute(query).fetchdf()

## 5. Composition of Lowest-Performing Schools

The lowest-performing schools are revisited with school-type included to evaluate whether particular structures are overrepresented among extreme outcomes.

This inspection helps distinguish between isolated low values ans systematic patterns.

In [ ]:
query = """
select
    school_type,
    district_number,
    district_name,
    school_number,
    school_name,
    graduation_rate_2023_24
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
order by graduation_rate_2023_24 asc
limit 15;
"""

con.execute(query).fetchdf()

## 6. Distributional Spread by School Type

To better understand outcome variability, minimum, maximum, and average graduation rates are computed by school type.

This comparison highlights differences in spread and tail behavior across school categories.

In [ ]:
query = """
select
    school_type,
    avg(graduation_rate_2023_24) as avg_grad_rate,
    min(graduation_rate_2023_24) as min_grad_rate,
    max(graduation_rate_2023_24) as max_grad_rate,
    count(*) as n_schools
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type
order by school_type;
"""

con.execute(query).fetchdf()

## 7. Threshold-Based Exploration

Graduation rates are evaluated against fixed benchmarks to explore how many schools fall below commonly referenced performance thresholds.

Counts below 80% and 70% graduation rates are examined by school type to support early risk framing.

In [ ]:
query = """
select
    school_type,
    count(*) as n_below_80
from main.stg_fl_school_grades
where graduation_rate_2023_24 < 80
group by school_type
order by n_below_80 desc;
"""

con.execute(query).fetchdf()

In [ ]:
query = """
select
    school_type,
    count(*) as n_below_70
from main.stg_fl_school_grades
where graduation_rate_2023_24 < 70
group by school_type;
"""

con.execute(query).fetchdf()

## 8. Proportional Representation Among Low Performers

To contextualize threshold counts, the proportion of low-performing schools attributed to each school type is calculated.

This step distinguishes absolute counts form relative representation.

In [ ]:
query = """
select
    school_type,
    count(*) * 1.0 /
        (select count(*) from main.stg_fl_school_grades where graduation_rate_2023_24 < 80)
        as proportion_of_low_performers
from main.stg_fl_school_grades
where graduation_rate_2023_24 < 80
group by school_type
"""

con.execute(query).fetchdf()

## 9. Graduation Rates and Title I Status

Graduation outcomes are further stratified by Title I staus to examine whether economic designation interacts with school structure.

This comparison supports later feature selection and risk signal evaluation.

In [ ]:
query = """
select
    school_type,
    title_i,
    avg(graduation_rate_2023_24) as avg_grad_rate,
    count(*) as n
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type, title_i
order by school_type, title_i;
"""

con.execute(query).fetchdf()

## 10. Socioeconomic Context and Graduation Outcomes

Average socioeconomic disadvantage is compared alongside graduation rates to explore broad contextual relationships.

Both aggregate comparisons and correlation analysis are used to assess the direction and strength of association.

In [ ]:
query = """
select
    school_type,
    round(avg(pct_econ_disadvantaged), 1) as avg_pct_econ,
    round(avg(graduation_rate_2023_24), 2) as avg_grad_rate
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type;
"""

con.execute(query).fetchdf()

## 11. Socioeconomic Disadvantage and Graduation Outcomes

To quantify the relationship between socioeconomic context and graduation outcomes, the correlation between the percentage of economically disadvantaged students and graduation rate is computed across all schools with available outcomes.

This provides a directional measure of association to support downstream feature evaluation.

In [ ]:
query = """
select
    corr(pct_econ_disadvantaged, graduation_rate_2023_24) as correlation
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null;
"""

con.execute(query).fetchdf()

## 12. Correlation Stratified by School Type

The relationship between socioeconomic disadvantage and graduation outcomes is further examined within school types.

Computing correlations separately by school type helps determine whether the strength of association varies across structural categories.

In [ ]:
query = """
select
    school_type,
    corr(pct_econ_disadvantaged, graduation_rate_2023_24) as correlation
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type;
"""

con.execute(query).fetchdf()

## 13. Preliminary Graduation Risk Prevalence

To support later risk modeling, a preliminary threshold-based definition of graduation risk is explored.

Schools with graduation rates below 80% are flagged as potentially at risk, and the overall prevalence of such schools is computed.

In [ ]:
query = """
select
    count(*) as total_schools,
    sum(case when graduation_rate_2023_24 < 80 then 1 else 0 end) as n_at_risk
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null;
"""

con.execute(query).fetchdf()

## 14. Graduation Risk Distribution by School Type

Finally, the distribution of at-risk schools is examined by school type to assess whether risk prevalence differs across structural categories.

This breakdown informs both feature selection and potential stratified modeling approaches.

In [ ]:
query = """
select
    school_type,
    count(*) as n_total,
    sum(case when graduation_rate_2023_24 < 80 then 1 else 0 end) as n_at_risk
from main.stg_fl_school_grades
where graduation_rate_2023_24 is not null
group by school_type;
"""

con.execute(query).fetchdf()

In [ ]:
con.close()